In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity

In [15]:
rating = pd.read_csv("./ml-100k/u.data")                                                    # user id | item id | rating | timestamp
movie = pd.read_csv("./ml-100k/u.item", sep="|", encoding="latin-1", header=None)           # movie id | movie title | release date | video release date | IMDb URL | unknown | Action | Adventure | Animation | etc.
user = pd.read_csv("./ml-100k/u.user")                                                      # user id | age | gender | occupation | zip code

In [16]:
# 1. Choose a user from the MovieLens 100K dataset.

def get_user_id():
    user_id = int(input("Enter a user ID: "))

    if user_id in user[0].values:
        return user_id
    else:
        print("Invalid user ID. Please try again.")
        return get_user_id()

In [17]:
# 2. Identify the movies that user rated and the movies they have not yet rated, using the movie IDs in u.data.

def get_user_movies(user_id):
    rated_movies = []
    unrated_movies = []

    # if user id = user_id, add movie id to rated_movies
    for i in range(ratings.shape[0]):
        if ratings.iloc[i, 0] == user_id:
            rated_movies.append(ratings.iloc[i, 1])

    # if movie id not in rated_movies, add to unrated_movies
    for i in range(movies.shape[0]):
        movie_id = movies.iloc[i, 0]
        if movie_id not in rated_movies:
            unrated_movies.append(movie_id)

    return rated_movies, unrated_movies

In [18]:
# 3. Of the movies that were rated, identify the ones that were rated positively via an inclusive threshold. Ratings range from 1 - 5.

def get_positive_movies(rated_movies, threshold):
    if threshold < 0 or threshold > 5:
        print("Error: Threshold must be between 0 and 5.")
        return
        
    positive_movies = []

    for movie_id in rated_movies:
        for i in range(ratings.shape[0]):
            if ratings.iloc[i, 1] == movie_id and ratings.iloc[i, 2] >= threshold:
                positive_movies.append(movie_id)
                break

    return positive_movies

In [19]:
# 4. Retrieve the genre vector for each positively rated movie and each unrated movie from u.item.

def get_genre_vectors(positive_movies, unrated_movies):
    positive_vectors = []
    unrated_vectors = []

    for movie_id in positive_movies:
        movie = movies[movies[0] == movie_id]
        genre_vector = []

        for i in range(5, 24):
            genre_vector.append(movie.iloc[0, i])

        positive_vectors.append(genre_vector)

    for movie_id in unrated_movies:
        movie = movies[movies[0] == movie_id]
        genre_vector = []

        for i in range(5, 24):
            genre_vector.append(movie.iloc[0, i])

        unrated_vectors.append(genre_vector)

    return positive_vectors, unrated_vectors


In [20]:
# 4. For each unseen movie, calculate its similarity to every positively rated movie, then average those similarities into one score.

def get_similarity_scores(positive_vectors, unrated_vectors):
    similarity_matrix = cosine_similarity(positive_vectors, unrated_vectors)
    # Each row represents a positively rated movie
    # Each column represents an unrated movie
    # Each value is the cosine similarity between that pair of movies

    similarity_scores = []

    for i in range(similarity_matrix.shape[1]):
        total = 0

        for j in range(similarity_matrix.shape[0]):
            total += similarity_matrix[j, i]

        average = total / similarity_matrix.shape[0]
        similarity_scores.append(average)

    return similarity_scores

In [ ]:
# cross validation using rated movies?

In [ ]:
# 5. Recommend the top (num_recommendations) unrated movies with the highest average similarity scores.

# Movie IDs and similarity scores are linked by index
# Pair them together so their association is preserved during selection sort

def recommend_movies(unrated_movies, similarity_scores, num_recommendations):
    recommendations = []

    for i in range(len(unrated_movies)):
        recommendations.append([unrated_movies[i], similarity_scores[i]])

    # Sort from highest similarity score to lowest
    for i in range(len(recommendations)):
        highest = i

        for j in range(i + 1, len(recommendations)):
            if recommendations[j][1] > recommendations[highest][1]:
                highest = j

        # Swap the movie-score pairs
        temp = recommendations[i]
        recommendations[i] = recommendations[highest]
        recommendations[highest] = temp

    print("Top reccomended movies:")
    for i in range(len(recommendations)):
        print(f"Recommendation {i + 1}: Movie ID {recommendations[i][0]}, Similarity Score: {recommendations[i][1]:.3f}")
    
    return recommendations[:num_recommendations]